# Group 16 Model Evaluation

This notebook is a clean evaluation notebook.

Use it to:
- choose any checkpoint from `models/`
- auto-detect the saved architecture from the checkpoint keys
- load the validation and test sets from `data_new/`
- find the best validation threshold
- evaluate the selected model on the test set

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torchvision.models import densenet121, efficientnet_b0, efficientnet_b3, vit_b_16

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.dataloader import get_dataloaders
from src.models.cnn_baseline import SimpleCNN
from src.models.cnn_batchnorm import BatchNormCNN
from src.models.cnn_batchnorm_deeper import DeeperBatchNormCNN
from src.models.cnn_batchnorm_residual import ResidualBatchNormCNN
from src.utils import find_best_threshold, evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
MODELS_DIR = ROOT / 'models'
TRAIN_CSV = ROOT / 'data_new/splits/train.csv'
VAL_CSV = ROOT / 'data_new/splits/val.csv'
TEST_CSV = ROOT / 'data_new/splits/test.csv'
TRAIN_IMAGE_DIR = ROOT / 'data_new/images/train'
TEST_IMAGE_DIR = ROOT / 'data_new/images/test'

BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 0

available_models = sorted(path.name for path in MODELS_DIR.glob('*.pth'))
if not available_models:
    raise FileNotFoundError(f'No .pth checkpoints found in {MODELS_DIR}')

display(pd.DataFrame({'available_models': available_models}))

# Change this to any checkpoint shown above.
SELECTED_MODEL = available_models[0]

print(f'Selected checkpoint: {SELECTED_MODEL}')

In [ ]:
def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        return checkpoint['state_dict']
    if isinstance(checkpoint, dict):
        return checkpoint
    raise TypeError(f'Unsupported checkpoint type: {type(checkpoint)}')


class BatchNormGapCNN(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


def detect_checkpoint_format(state_dict):
    keys = list(state_dict.keys())

    if any(key.startswith('model.class_token') for key in keys):
        return 'vit_wrapped'

    if 'class_token' in state_dict:
        return 'vit_raw'

    if any(key.startswith('stem.') for key in keys):
        return 'cnn_residual_batchnorm'

    if any(key.startswith('model.features.') for key in keys):
        if 'model.classifier.weight' in state_dict and 'model.classifier.bias' in state_dict:
            return 'densenet_baseline'
        if 'model.classifier.1.running_mean' in state_dict:
            return 'densenet_batchnorm_weighted'
        if 'model.classifier.0.weight' in state_dict and 'model.classifier.3.weight' in state_dict:
            return 'densenet_dropout_style'

    if 'features.0.0.weight' in state_dict and 'classifier.0.weight' in state_dict:
        out_channels = int(state_dict['features.0.0.weight'].shape[0])
        if out_channels == 32:
            return 'efficientnet_b0'
        if out_channels == 40:
            return 'efficientnet_b3'

    if any(key.startswith('features.') for key in keys):
        first_conv = state_dict.get('features.0.weight')

        if first_conv is not None and 'classifier.2.weight' in state_dict and 'classifier.5.weight' in state_dict:
            return 'cnn_batchnorm_gap'

        if first_conv is not None and 'classifier.1.weight' in state_dict:
            classifier_in_features = int(state_dict['classifier.1.weight'].shape[1])
            has_batchnorm = any(key.startswith('features.1.running_') for key in keys)

            if has_batchnorm and classifier_in_features == 100352:
                return 'cnn_batchnorm'
            if has_batchnorm and classifier_in_features == 256:
                return 'cnn_deeper_batchnorm'
            if not has_batchnorm and classifier_in_features == 100352:
                return 'cnn_baseline'

    raise ValueError(
        'Could not detect a supported architecture from this checkpoint. '
        'If you added a new model family, extend detect_checkpoint_format() first.'
    )


def build_model_for_format(checkpoint_format):
    if checkpoint_format == 'cnn_baseline':
        return SimpleCNN(num_classes=1)

    if checkpoint_format == 'cnn_batchnorm':
        return BatchNormCNN(num_classes=1)

    if checkpoint_format == 'cnn_batchnorm_gap':
        return BatchNormGapCNN(num_classes=1)

    if checkpoint_format == 'cnn_deeper_batchnorm':
        return DeeperBatchNormCNN(num_classes=1)

    if checkpoint_format == 'cnn_residual_batchnorm':
        return ResidualBatchNormCNN(num_classes=1)

    if checkpoint_format == 'densenet_baseline':
        model = densenet121(weights=None)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, 1)
        return model

    if checkpoint_format == 'densenet_batchnorm_weighted':
        model = densenet121(weights=None)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )
        return model

    if checkpoint_format == 'densenet_dropout_style':
        model = densenet121(weights=None)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
        )
        return model

    if checkpoint_format == 'efficientnet_b0':
        model = efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Linear(in_features, 1))
        return model

    if checkpoint_format == 'efficientnet_b3':
        model = efficientnet_b3(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Linear(in_features, 1))
        return model

    if checkpoint_format == 'vit_wrapped':
        model = vit_b_16(weights=None)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, 1)
        return model

    if checkpoint_format == 'vit_raw':
        model = vit_b_16(weights=None)
        in_features = model.heads.head.in_features
        model.heads = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, 1))
        return model

    raise ValueError(f'Unsupported checkpoint format: {checkpoint_format}')


def normalize_state_dict_for_model(state_dict, checkpoint_format):
    if checkpoint_format.startswith('densenet_'):
        return {key.replace('model.', '', 1): value for key, value in state_dict.items()}

    if checkpoint_format == 'vit_wrapped':
        return {key.replace('model.', '', 1): value for key, value in state_dict.items()}

    return state_dict


def load_model_from_checkpoint(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    raw_state_dict = extract_state_dict(checkpoint)
    checkpoint_format = detect_checkpoint_format(raw_state_dict)
    model = build_model_for_format(checkpoint_format)
    state_dict = normalize_state_dict_for_model(raw_state_dict, checkpoint_format)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as error:
        raise RuntimeError(
            f'Failed to load {checkpoint_path.name} as {checkpoint_format}.\n{error}'
        ) from error

    model = model.to(device)
    model.eval()

    return model, checkpoint_format


In [ ]:
checkpoint_path = MODELS_DIR / SELECTED_MODEL
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

_, val_loader, test_loader = get_dataloaders(
    train_csv=str(TRAIN_CSV),
    val_csv=str(VAL_CSV),
    test_csv=str(TEST_CSV),
    image_dir=str(TRAIN_IMAGE_DIR),
    test_image_dir=str(TEST_IMAGE_DIR),
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    num_workers=NUM_WORKERS,
)

model, checkpoint_format = load_model_from_checkpoint(checkpoint_path, device)

print(f'Loaded checkpoint: {checkpoint_path.name}')
print(f'Detected architecture: {checkpoint_format}')
print(f'Validation batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

In [ ]:
best_threshold, best_f2 = find_best_threshold(model, val_loader, device)
print(f'Chosen threshold for test evaluation: {best_threshold:.2f}')

In [ ]:
evaluate_model(model, test_loader, device, threshold=best_threshold)